# Neural Compute Fabric — Interactive Architecture Laboratory

**Private evaluation prototype · Copyright © 2026 · All rights reserved.** See `NOTICE.md` in the repository. This notebook explains and exercises the actual graph engine instead of inventing a separate implementation.

**Objectives:** inspect a programmable computational DAG; ingest structured IoT readings; run Pydantic/NumPy transformations; inspect softmax/stochastic routing; attach an example ML-like function; simulate a continuous input stream; and evaluate the relationship between data characteristics and runtime costs.

**Limitations:** routing weights are not learned in this example; the risk score is not calibrated failure probability; C++/CUDA/PyTorch backends are pluggable callables, not included trained deployments. Never treat demonstration scores as safety decisions.

## Setup

Open this notebook from the repository root or `notebooks/`. Install core dependencies using `pip install -r requirements.txt` in a terminal. The cell below locates `engine.py` from either location.

In [ ]:
from pathlib import Path
import sys, json, math, time
import numpy as np


In [ ]:
project = Path.cwd()
if not (project / 'engine.py').exists():
    project = project.parent
assert (project / 'engine.py').exists(), 'Open this notebook from the repo root or notebooks/.'
sys.path.insert(0, str(project))
from engine import Graph, GraphSpec, events
print('Project:', project)
print('NumPy:', np.__version__)

## 1 · Inspect a real configuration-defined graph

In [ ]:
config = json.loads((project / 'examples' / 'iot_graph.json').read_text())
spec = GraphSpec.model_validate(config)
engine = Graph(spec, seed=42)
for name in engine.order:
    node = spec.nodes[name]
    print(f'{name:20} ← {", ".join(node.inputs) or "event"} [{node.kind}]')

## 2 · Run the sample IoT events and examine routing

In [ ]:
iot_events = list(events(str(project / 'examples' / 'iot_events.jsonl'), stream=False))
outputs = []
for event in iot_events:
    try:
        result = engine.run(event)
        outputs.append(result)
        print('event_id:', event.get('event_id'), '\n', json.dumps(result, indent=2))
    except (ValueError, KeyError, TypeError) as exc:
        print('Rejected event:', event.get('event_id'), str(exc))

## 3 · NumPy as a functional routing node

Softmax turns routing scores into normalized weights. It does **not** automatically learn which sensor predicts failure best. Compare deterministic routing with seeded stochastic sampling.

In [ ]:
features = np.array([1.5, 1.6, 1.0])
exponentials = np.exp(features - features.max())
weights = exponentials / exponentials.sum()
print('Weights:', weights.round(4), 'sum:', weights.sum())
print('Weighted signal:', np.dot(features, weights))
rng = np.random.default_rng(42)
choices = rng.choice(['temperature', 'vibration', 'voltage'], size=1000, p=weights)
print('Sampled selection proportions:', {name: round(np.mean(choices == name), 3) for name in ['temperature', 'vibration', 'voltage']})

## 4 · Change the graph from Python (without editing the engine)

In [ ]:
modified = json.loads(json.dumps(config))
modified['nodes']['routing_weights']['params']['temperature'] = 0.3
modified['nodes']['selected_sensor']['params']['strategy'] = 'argmax'
modified['outputs'] = ['routing_weights', 'selected_sensor', 'risk_score']
modified_engine = Graph(GraphSpec.model_validate(modified), seed=42)
print(json.dumps(modified_engine.run(iot_events[0]), indent=2))

## 5 · Register an external inference function

`torch_predict`, `sklearn_predict`, and `native` are *adapter types*. We register a small deterministic placeholder here. To plug in a trained PyTorch model or a native C++ extension, replace the registered callable; do not mistake this example for trained inference.

In [ ]:
model_spec = GraphSpec.model_validate({
    'nodes': {
        'features': {'kind':'input','params':{'field':'features'}},
        'inference': {'kind':'torch_predict', 'inputs':['features'],
                      'params':{'registry_key':'demo_predictor'}}
    }, 'outputs':['inference']
})
def demo_predictor(features):
    arr = np.asarray(features, dtype=float)
    return float(1 / (1 + np.exp(-np.mean(arr))))
model_graph = Graph(model_spec, registry={'demo_predictor': demo_predictor})
print(model_graph.run({'features':[0.2, 0.4, 0.6]}))

## 6 · Simulate live streaming IoT readings

In [ ]:
def simulated_sensor_stream(n=12, seed=7):
    rng = np.random.default_rng(seed)
    for index in range(n):
        yield {'event_id': f'sim-{index:03d}', 'reading': {
            'temperature': float(rng.normal(65, 4)),
            'vibration': float(max(0, rng.normal(0.5, 0.12))),
            'voltage': float(max(0, rng.normal(220, 6)))
        }}

stream_graph = Graph(spec, seed=42)
stream_results = []
for event in simulated_sensor_stream():
    output = stream_graph.run(event)
    stream_results.append({'id': event['event_id'],
                           'routing_index': output['selected_sensor']['index'],
                           'illustrative_risk_score': output['risk_score']})
print(*stream_results[:5], sep='\n')

## 7 · Compute-planning starting point

Raw feature throughput is only one component of resource planning. It does **not** imply GPU necessity or accurately predict latency. Model FLOPs, windows, transfer overhead, buffering, and benchmarks are separate inputs.

In [ ]:
events_per_second = 100_000
features_per_event = 128
float_bytes = 4
raw_MB_per_second = events_per_second * features_per_event * float_bytes / 1_000_000
print(f'Raw float32 feature throughput: {raw_MB_per_second:,.1f} MB/s')
print(f'Raw 60-second feature window: {raw_MB_per_second*60/1000:,.2f} GB (decimal)')

## 8 · Validation and tests

The engine rejects cycles and nonfinite numeric inputs. The repository has further unit tests.

In [ ]:
import unittest
suite = unittest.defaultTestLoader.discover(str(project / 'tests'), pattern='test_*.py')
result = unittest.TextTestRunner(verbosity=1).run(suite)
assert result.wasSuccessful()

## Next development milestones

1. Define typed ports for input/output shapes and add explicit compatibility checks.
2. Add source connectors, event timestamps, partitioning, backpressure, and sensor identity.
3. Add a safe graph search algorithm to propose candidate feature-to-model edges.
4. Compare candidate graphs against **time-ordered** held-out IoT data.
5. Benchmark CPU versus GPU runtimes and collect per-node execution traces.
6. Add optional PyTorch training adapters and native C++/CUDA kernels with proper gradient definitions where needed.
7. Deploy selected graphs with FastAPI, Helm, Kubernetes, and GitOps, with controls and observability.

**IP note:** Keep sensitive algorithms and configuration private. Third-party libraries retain their own licenses. No patent or ownership assertion is established solely by this notebook.